In [1]:
import pandas as pd
from pathlib import Path
import json
from typing import Set
from datetime import datetime
import os

In [2]:
base_path = '/Users/pena/nl2c_projetcts/text2rules/data'
dataset_names = ['sisu', 'hospital-holo', 'tax-chu', 'restaurant-inspections']
dataset_names = ['tax-chu','restaurant-inspections']
dataset_names = ['restaurant-inspections']

In [3]:
def extract_rules(file_path: str, dataset_name) -> Set[str]:
    """
    Reads a JSON file with rule mappings and extracts all unique column names
    from the rules predicates.

    Args:
        file_path: Path to the JSON file

    Returns:
        Set of unique column names found in the rules
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    all_dcs = []
    tuple_pair_dcs = []

    # we use the first letter of the dataset name as the tuple identifier: e.g., sisu->s1,s2; tax-chu->t1,t2 ...
    t1_str = dataset_name[0] + "1"
    t2_str = dataset_name[0] + "2"

    non_tuple_pair_symbols = {'LIKE', 'NOT LIKE', 'IN',
                              'NOT IN', 'BETWEEN', 'IS NULL', 'IS NOT NULL'}

    # Iterate through all mappings
    for item in data.get('mapping', []):
        rules = item.get('rules')

        # Skip if rules is null or not applicable
        if rules is None:
            continue

        # Each rule is an array of predicates
        for rule in rules:

            predicate_set = set()
            is_tuple_pair = True

            # First pass: check if ALL predicates in the rule are tuple-pair
            for predicate in rule:
                predicate = predicate.strip()

                if not (t1_str in predicate and t2_str in predicate):
                    is_tuple_pair = False
                
                predicate = predicate.replace(
                    t1_str, 't1').replace(t2_str, 't2')
                
                
                predicate_set.add(predicate)

            # Always add to all_dcs
            all_dcs.append(predicate_set)

            # Only add to tuple_pair_dcs if the entire rule qualifies
            if is_tuple_pair:
                tuple_pair_dcs.append(predicate_set)

    return tuple_pair_dcs, all_dcs

In [4]:
def normalize_nsymbols(df):
    """Add a normalized NSymbols column where higher is better."""
    min_nsymbols = df['NSymbols'].min()
    df['NSymbols_norm'] = min_nsymbols / df['NSymbols']
    return df


def compute_mixed_score(df, weight_coverage=0.5):
    """
    Compute a mixed score combining Coverage and normalized NSymbols.
    weight_coverage: weight for coverage (0 to 1), the rest goes to NSymbols_norm
    """
    if 'NSymbols_norm' not in df.columns:
        df = normalize_nsymbols(df)
    
    df['Mixed_score'] = weight_coverage * df['Coverage'] + (1 - weight_coverage) * df['NSymbols_norm']
    return df


def rank_dcs(df, sort_by='Coverage', ascending=False):
    """
    Rank DCs by sorting on the specified column.
    
    Args:
        df: DataFrame with DC results
        sort_by: Column name to sort by
        ascending: Sort order (False = highest first, which is default for better scores)
    
    Returns:
        Sorted DataFrame
    """
    return df.sort_values(sort_by, ascending=ascending)


def normalize_predicate(pred):
    """
    Comprehensively normalize a predicate for comparison:
    - Normalize tuple identifiers: any letter+digit (h1, r1, s1) -> t1/t2
    - Convert to lowercase
    - Normalize SQL operators and keywords
    - Normalize numeric values (13.0 -> 13)
    - Remove extra spaces
    
    Returns:
        normalized predicate string
    """
    import re
    
    # Convert to lowercase first
    pred = pred.lower()
    
    # Replace any single-letter + digit patterns with t + digit
    # h1 -> t1, r1 -> t1, s1 -> t1, etc.
    # h2 -> t2, r2 -> t2, s2 -> t2, etc.
    pred = re.sub(r'\b[a-z]1\.', 't1.', pred)
    pred = re.sub(r'\b[a-z]2\.', 't2.', pred)
    
    # Normalize operators BEFORE removing spaces
    pred = pred.replace('<>', '!=')
    pred = pred.replace('< >', '!=')
    pred = pred.replace('==', '=')
    pred = pred.replace('= =', '=')
    
    # Normalize SQL keywords
    pred = pred.replace(' not like ', ' notlike ')
    pred = pred.replace(' not in ', ' notin ')
    
    # Normalize spacing around operators
    pred = re.sub(r'([<>=!]+)', r' \1 ', pred)
    pred = re.sub(r'\s+', ' ', pred)  # Collapse multiple spaces
    pred = pred.strip()
    
    # Remove all spaces for consistent comparison
    pred = pred.replace(' ', '')
    
    # Normalize numeric values: remove .0 from whole numbers (13.0 -> 13, 365.0 -> 365)
    pred = re.sub(r'(\d+)\.0\b', r'\1', pred)
    
    return pred


def parse_dc_to_predicates(dc_string):
    """
    Parse a DC string into a set of normalized predicates.
    Works with DC Discovery format: "not(t1.x=t2.x and t1.y>t2.y)"
    
    Returns:
        Set of normalized predicates
    """
    # Remove "not(" prefix and trailing ")"
    dc_string = dc_string.strip()
    if dc_string.startswith('not('):
        dc_string = dc_string[4:]
    if dc_string.endswith(')'):
        dc_string = dc_string[:-1]
    
    # Split by " and " to get individual predicates
    predicates = dc_string.split(' and ')
    
    # Normalize each predicate
    normalized_predicates = {normalize_predicate(pred.strip()) for pred in predicates}
    
    return normalized_predicates


def compute_dc_metrics(ground_truth_dcs, discovered_dcs):
    """
    Compute precision, recall, and F1 score for DC discovery.
    
    Args:
        ground_truth_dcs: List of sets of predicates (ground truth) - already normalized
        discovered_dcs: List of sets of predicates (discovered) - already normalized
    
    Returns:
        dict with 'precision', 'recall', 'f1', 'tp', 'fp', 'fn'
    """
    # Ground truth should already be normalized, but ensure consistency
    gt_normalized = []
    for dc in ground_truth_dcs:
        normalized_dc = {normalize_predicate(pred) for pred in dc}
        gt_normalized.append(normalized_dc)
    
    # Discovered DCs should also be normalized (they come from parsers)
    # But we'll ensure they're normalized here too for safety
    disc_normalized = []
    for dc in discovered_dcs:
        # If dc is already a set of strings, normalize each
        if isinstance(dc, set):
            normalized_dc = {normalize_predicate(pred) if isinstance(pred, str) else pred for pred in dc}
            disc_normalized.append(normalized_dc)
        else:
            disc_normalized.append(dc)
    
    # Convert to sets of frozensets for comparison
    gt_set = {frozenset(dc) for dc in gt_normalized}
    disc_set = {frozenset(dc) for dc in disc_normalized}
    
    # Calculate metrics
    tp = len(gt_set & disc_set)  # Intersection: true positives
    fp = len(disc_set - gt_set)   # Discovered but not in GT: false positives
    fn = len(gt_set - disc_set)   # In GT but not discovered: false negatives
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'gt_count': len(gt_set),
        'discovered_count': len(disc_set)
    }

In [5]:


results_path = Path('results_dc_discovery')
output_dir = Path('dc_discovery_baseline_results')
output_dir.mkdir(exist_ok=True)

# Create timestamp for filename
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f'evaluation_results_{timestamp}.csv'

approx_levels = [0.001, 0.005, 0.01, 0.05, 0.1]
all_results = []

# Initialize main_results list to collect best results across all methods
main_results = []

for name in dataset_names:
    print(f"\nProcessing dataset: {name}")
    
    # Load ground truth
    dataset_dir = Path(base_path) / name
    json_file_name = dataset_dir / f"{name}_text2rule_gt.json"
    
    if not json_file_name.exists():
        print(f"Warning: Ground truth not found for {name}")
        continue
    
    tuple_pair_dcs, all_dcs = extract_rules(json_file_name, name)
    
    # Ranking strategies
    ranking_strategies = [
        ('Coverage', 'Coverage', False),
        ('NSymbols_norm', 'NSymbols_norm', False),
        ('Mixed_score', 'Mixed_score', False)
    ]

    for dc in all_dcs:
        print(dc)
    
    # Process EXACT DCs
    exact_files = list(results_path.glob(f"{name}_trimmed_exact*.csv"))
    if exact_files:
        df_exact_dcs = pd.read_csv(exact_files[0])
        df_exact_dcs = normalize_nsymbols(df_exact_dcs)
        df_exact_dcs = compute_mixed_score(df_exact_dcs, weight_coverage=0.5)
        discovered_predicates = [parse_dc_to_predicates(dc) for dc in df_exact_dcs['DC']]
        
        top_ks = [10, 20, 50, 100, len(df_exact_dcs)]
        
        for strategy_name, sort_column, ascending in ranking_strategies:
            ranked_df = rank_dcs(df_exact_dcs.copy(), sort_by=sort_column, ascending=ascending)
            
            for top_k in top_ks:
                if top_k > len(ranked_df):
                    continue
                
                top_k_indices = ranked_df.head(top_k).index
                top_k_discovered = [discovered_predicates[i] for i in top_k_indices]
                metrics = compute_dc_metrics(all_dcs, top_k_discovered)
                
                all_results.append({
                    'dataset': name,
                    'method': strategy_name,
                    'dc_type': 'exact',
                    'approximation': 0.0,
                    'top_k': top_k if top_k < len(df_exact_dcs) else 'all',
                    'precision': metrics['precision'],
                    'recall': metrics['recall'],
                    'f1': metrics['f1'],
                    'tp': metrics['tp'],
                    'fp': metrics['fp'],
                    'fn': metrics['fn'],
                    'gt_count': metrics['gt_count'],
                    'discovered_count': metrics['discovered_count'],
                    'best_exact': False,
                    'best_approximate': False,
                    'best_overall': False
                })
    
    # Process APPROXIMATE DCs
    for approx_level in approx_levels:
        approx_files = list(results_path.glob(f"{name}_trimmed_approx_{approx_level:.5f}*.csv"))
        if not approx_files:
            continue
        
        df_approx_dcs = pd.read_csv(approx_files[0])
        df_approx_dcs = normalize_nsymbols(df_approx_dcs)
        df_approx_dcs = compute_mixed_score(df_approx_dcs, weight_coverage=0.5)
        approx_predicates = [parse_dc_to_predicates(dc) for dc in df_approx_dcs['DC']]
        
        top_ks = [10, 20, 50, 100, len(df_approx_dcs)]
        
        for strategy_name, sort_column, ascending in ranking_strategies:
            ranked_df = rank_dcs(df_approx_dcs.copy(), sort_by=sort_column, ascending=ascending)
            
            for top_k in top_ks:
                if top_k > len(ranked_df):
                    continue
                
                top_k_indices = ranked_df.head(top_k).index
                top_k_discovered = [approx_predicates[i] for i in top_k_indices]
                metrics = compute_dc_metrics(all_dcs, top_k_discovered)
                
                all_results.append({
                    'dataset': name,
                    'method': strategy_name,
                    'dc_type': 'approximate',
                    'approximation': approx_level,
                    'top_k': top_k if top_k < len(df_approx_dcs) else 'all',
                    'precision': metrics['precision'],
                    'recall': metrics['recall'],
                    'f1': metrics['f1'],
                    'tp': metrics['tp'],
                    'fp': metrics['fp'],
                    'fn': metrics['fn'],
                    'gt_count': metrics['gt_count'],
                    'discovered_count': metrics['discovered_count'],
                    'best_exact': False,
                    'best_approximate': False,
                    'best_overall': False
                })

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

# Mark best rows for each dataset
for dataset in results_df['dataset'].unique():
    dataset_mask = results_df['dataset'] == dataset
    
    # Mark best exact
    exact_mask = dataset_mask & (results_df['dc_type'] == 'exact')
    if exact_mask.sum() > 0:
        best_exact_idx = results_df[exact_mask]['f1'].idxmax()
        results_df.at[best_exact_idx, 'best_exact'] = True
    
    # Mark best approximate
    approx_mask = dataset_mask & (results_df['dc_type'] == 'approximate')
    if approx_mask.sum() > 0:
        best_approx_idx = results_df[approx_mask]['f1'].idxmax()
        results_df.at[best_approx_idx, 'best_approximate'] = True
    
    # Mark best overall (across both exact and approximate)
    if dataset_mask.sum() > 0:
        best_overall_idx = results_df[dataset_mask]['f1'].idxmax()
        results_df.at[best_overall_idx, 'best_overall'] = True

# Save to CSV
# results_df.to_csv(output_file, index=False)
# print(f"\nResults saved to: {output_file}")
# print(f"Total rows: {len(results_df)}")

# Display summary
print("\nSummary by dataset:")
for dataset in results_df['dataset'].unique():
    dataset_df = results_df[results_df['dataset'] == dataset]
    print(f"\n{dataset}:")
    
    # Best exact
    best_exact = dataset_df[dataset_df['best_exact'] == True]
    if not best_exact.empty:
        row = best_exact.iloc[0]
        print(f"  Best Exact: {row['method']}, top-{row['top_k']}, F1={row['f1']:.4f}")
    
    # Best approximate
    best_approx = dataset_df[dataset_df['best_approximate'] == True]
    if not best_approx.empty:
        row = best_approx.iloc[0]
        print(f"  Best Approximate: {row['method']} @ {row['approximation']}, top-{row['top_k']}, F1={row['f1']:.4f}")
    
    # Best overall
    best_overall = dataset_df[dataset_df['best_overall'] == True]
    if not best_overall.empty:
        row = best_overall.iloc[0]
        print(f"  Best Overall: {row['dc_type']}, {row['method']} @ {row['approximation']}, top-{row['top_k']}, F1={row['f1']:.4f}")

# Collect main results - Best Exact and Best Approximate

for dataset in results_df['dataset'].unique():
    # Best exact
    dataset_exact = results_df[(results_df['dataset'] == dataset) & 
                               (results_df['best_exact'] == True)]
    if not dataset_exact.empty:
        row = dataset_exact.iloc[0]
        main_results.append({
            'dataset': dataset,
            'method': 'DCDiscovery-Exact',
            'precision': row['precision'],
            'recall': row['recall'],
            'f1': row['f1']
        })
    
    # Best approximate
    dataset_approx = results_df[(results_df['dataset'] == dataset) & 
                                (results_df['best_approximate'] == True)]
    if not dataset_approx.empty:
        row = dataset_approx.iloc[0]
        main_results.append({
            'dataset': dataset,
            'method': 'DCDiscovery-Approximate',
            'precision': row['precision'],
            'recall': row['recall'],
            'f1': row['f1']
        })



Processing dataset: restaurant-inspections
{"t1.action NOT LIKE '%Closed by DOHMH%'", 't1.consecutive_high_scores >= 3'}
{"t1.grade != 'A'", 't1.score <= 13', 't1.score >= 0'}
{'t1.score <= 27', 't1.score >= 14', "t1.grade != 'B'"}
{'t1.score >= 28', "t1.grade != 'C'"}
{'t1.days_to_next_inspection > 365'}
{'t1.score <= 13', 't1.score >= 0', "t1.inspection_type LIKE '%Initial Inspection%'", "t1.grade = 'A'", 't1.grade_date != t1.inspection_date'}
{"t1.inspection_type LIKE '%Re-inspection%'", "t1.grade = 'A'", 't1.grade_date != t1.inspection_date'}
{"t1.inspection_type LIKE '%Re-inspection%'", "t1.grade IN ('B', 'C')", 't1.grade_date != t1.inspection_date'}
{'t1.camis = t2.camis', "t2.grade IN ('B', 'C')", "t1.grade = 'A'", 't1.days_to_next_inspection < t2.days_to_next_inspection'}

Summary by dataset:

restaurant-inspections:
  Best Exact: Coverage, top-10, F1=0.0000
  Best Approximate: Coverage @ 0.01, top-10, F1=0.0000
  Best Overall: exact, Coverage @ 0.0, top-10, F1=0.0000


In [6]:

def parse_dc_to_predicates_for_llm_baseline(dc_string):
    """
    Parse a DC string from LLM/Norma baseline into a set of normalized predicates.
    
    Handles multiple formats:
    - LLM format: "t1.areacode = t2.areacode AND t1.phone = t2.phone"
    - Norma format: "r1.score >= 0.0 AND r1.score <= 13.0"
    Note: Uses uppercase "AND" and doesn't have "not(...)" wrapper
    
    Returns:
        Set of normalized predicates
    """
    dc_string = dc_string.strip()
    
    # Split by uppercase " AND " to get individual predicates
    predicates = dc_string.split(' AND ')
    
    # Normalize each predicate (handles tuple identifiers, operators, spacing, etc.)
    normalized_predicates = {normalize_predicate(pred.strip()) for pred in predicates}
    
    return normalized_predicates


In [7]:
## load llm_prompt baseline results
llm_baseline_path = Path('/Users/pena/nl2c_projetcts/text2rules/results/llm_prompt_baseline/')

# output_dir already created in Cell 6
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f'llm_baseline_results_{timestamp}.csv'

llm_baseline_results = []

# Need to reload ground truth for each dataset
for dataset in dataset_names:
    print(f"\nProcessing LLM baseline for dataset: {dataset}")
    
    # Load ground truth
    dataset_dir = Path(base_path) / dataset
    json_file_name = dataset_dir / f"{dataset}_text2rule_gt.json"
    
    if not json_file_name.exists():
        print(f"Warning: Ground truth not found for {dataset}")
        continue
    
    tuple_pair_dcs, all_dcs = extract_rules(json_file_name, dataset)
    
    # Find all CSV files in the LLM baseline path
    llm_csv_files = list(llm_baseline_path.rglob('*.csv'))
    for csv_file in llm_csv_files:
        if dataset in csv_file.name:
            # Extract components from path
            parts = csv_file.parts
            baseline_idx = parts.index('llm_prompt_baseline')
            
            model = parts[baseline_idx + 1]
            mode = parts[baseline_idx + 2]
            prompt = parts[baseline_idx + 3]
            
            print(f"  Processing: {model}/{mode}/{prompt}")
            
            df_llm = pd.read_csv(csv_file)
            if 'dc' in df_llm.columns:
                dc_list = df_llm['dc'].dropna().tolist()
            else:
                dc_list = []

            if not dc_list:
                print(f"    No DCs found in {csv_file.name}")
                continue

            # Parse all DCs from this file
            discovered_predicates = []
            for dc in dc_list:
                try:
                    predicates = parse_dc_to_predicates_for_llm_baseline(str(dc))
                    discovered_predicates.append(predicates)
                except Exception as e:
                    print(f"    Error parsing DC: {dc[:50]}... - {e}")
                    continue


            for dc in discovered_predicates:
                print(dc)
            
            # Compute metrics for all discovered DCs
            metrics = compute_dc_metrics(all_dcs, discovered_predicates)
            
            llm_baseline_results.append({
                'dataset': dataset,
                'model': model,
                'mode': mode,
                'prompt': prompt,
                'method': f"{model}_{mode}_{prompt}",
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1'],
                'tp': metrics['tp'],
                'fp': metrics['fp'],
                'fn': metrics['fn'],
                'gt_count': metrics['gt_count'],
                'discovered_count': len(discovered_predicates),
                'best_f1': False
            })

# Convert to DataFrame
llm_results_df = pd.DataFrame(llm_baseline_results)

# Mark best F1 for each dataset and mode
for dataset in llm_results_df['dataset'].unique():
    for mode in llm_results_df['mode'].unique():
        mask = (llm_results_df['dataset'] == dataset) & (llm_results_df['mode'] == mode)
        if mask.sum() > 0:
            best_idx = llm_results_df[mask]['f1'].idxmax()
            llm_results_df.at[best_idx, 'best_f1'] = True

# Save to CSV
llm_results_df.to_csv(output_file, index=False)
print(f"\n{'='*60}")
print(f"LLM Baseline Results saved to: {output_file}")
print(f"Total rows: {len(llm_results_df)}")

# Display summary
print(f"\n{'='*60}")
print("Summary by dataset:")
for dataset in llm_results_df['dataset'].unique():
    dataset_df = llm_results_df[llm_results_df['dataset'] == dataset]
    print(f"\n{dataset}:")
    print(f"  Total configurations: {len(dataset_df)}")
    
    for mode in dataset_df['mode'].unique():
        mode_df = dataset_df[dataset_df['mode'] == mode]
        print(f"\n  Mode: {mode}")
        print(f"    Best F1: {mode_df['f1'].max():.4f}")
        
        best_row = mode_df[mode_df['best_f1'] == True]
        if not best_row.empty:
            row = best_row.iloc[0]
            print(f"    Best: {row['method']}")
            print(f"      Precision: {row['precision']:.4f}, Recall: {row['recall']:.4f}, F1: {row['f1']:.4f}")
            print(f"      TP: {row['tp']}, FP: {row['fp']}, FN: {row['fn']}")
            print(f"      Discovered: {row['discovered_count']}, Ground Truth: {row['gt_count']}")
print(f"{'='*60}")

# Add LLM results to main_results
for dataset in llm_results_df['dataset'].unique():
    # Best all-at-once
    dataset_all_at_once = llm_results_df[(llm_results_df['dataset'] == dataset) & 
                                         (llm_results_df['mode'] == 'all-at-once') &
                                         (llm_results_df['best_f1'] == True)]
    if not dataset_all_at_once.empty:
        row = dataset_all_at_once.iloc[0]
        main_results.append({
            'dataset': dataset,
            'method': 'LLM-AllAtOnce',
            'precision': row['precision'],
            'recall': row['recall'],
            'f1': row['f1']
        })
    
    # Best sentence-by-sentence
    dataset_sentence = llm_results_df[(llm_results_df['dataset'] == dataset) & 
                                      (llm_results_df['mode'] == 'sentence-by-sentence') &
                                      (llm_results_df['best_f1'] == True)]
    if not dataset_sentence.empty:
        row = dataset_sentence.iloc[0]
        main_results.append({
            'dataset': dataset,
            'method': 'LLM-SentenceBySentence',
            'precision': row['precision'],
            'recall': row['recall'],
            'f1': row['f1']
        })


Processing LLM baseline for dataset: restaurant-inspections
  Processing: gpt-5/sentence-by-sentence/with_examples
{'t1.score<=13', 't1.score>=0', "t1.grade!='a'"}
{'t1.score<=27', 't1.score>=14', "t1.grade!='b'"}
{"t1.grade!='c'", 't1.score>=28'}
{'t1.score<=13', 't1.score>=0', 't1.inspection_order=1', "t1.grade!='a'"}
{"t1.grade!='a'", 't1.gradeisnotnull', 't1.inspection_order=1'}
{"t1.grade='a'", 't1.inspection_order>1', 't1.grade_dateisnull'}
{"t1.actionnotlike'%gradepending%'", "t1.gradein('b','c')", 't1.inspection_order>1', 't1.grade_dateisnull'}
  Processing: gpt-5/all-at-once/with_examples
{'t1.score<=13', 't1.score>=0', "t1.grade!='a'"}
{'t1.score<=27', 't1.score>=14', "t1.grade!='b'"}
{"t1.grade!='c'", 't1.score>=28'}
{'t1.score<=13', 't1.score>=0', 't1.inspection_order=1', "t1.grade!='a'"}

LLM Baseline Results saved to: dc_discovery_baseline_results/llm_baseline_results_20260206_072055.csv
Total rows: 2

Summary by dataset:

restaurant-inspections:
  Total configurations: 

In [8]:
## Load Norma algorithm results
norma_base_path = Path('/Users/pena/nl2c_projetcts/text2rules/results/norma/')

# output_dir already created in Cell 6
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = output_dir / f'norma_results_{timestamp}.csv'

norma_results = []

# Need to reload ground truth for each dataset
for dataset in dataset_names:
    print(f"\nProcessing Norma results for dataset: {dataset}")
    
    # Load ground truth
    dataset_dir = Path(base_path) / dataset
    json_file_name = dataset_dir / f"{dataset}_text2rule_gt.json"
    
    if not json_file_name.exists():
        print(f"Warning: Ground truth not found for {dataset}")
        continue
    
    tuple_pair_dcs, all_dcs = extract_rules(json_file_name, dataset)
    
    # Find all CSV files in the Norma results path
    norma_csv_files = list(norma_base_path.rglob(f'{dataset}_*.csv'))
    
    # Filter out _runs_summary.csv files
    norma_csv_files = [f for f in norma_csv_files if '_runs_summary' not in f.name]
    
    for csv_file in norma_csv_files:
        print(f"  Processing: {csv_file.name}")
        
        # Look for corresponding _runs_summary.csv in the same directory
        summary_file = csv_file.parent / '_runs_summary.csv'
        
        if not summary_file.exists():
            print(f"    Warning: No _runs_summary.csv found for {csv_file.name}")
            continue
        
        # Load summary to get model/prompt information
        df_summary = pd.read_csv(summary_file)
        
        # Find the row corresponding to this file
        file_row = df_summary[df_summary['filename'] == csv_file.name]
        
        if file_row.empty:
            print(f"    Warning: No summary entry found for {csv_file.name}")
            continue
        
        row_info = file_row.iloc[0]
        classifier_model = row_info['classifier_model']
        classifier_prompt = row_info['classifier_prompt']
        mapper_model = row_info['mapper_model']
        mapper_prompt = row_info['mapper_prompt']
        
        print(f"    Classifier: {classifier_model}/{classifier_prompt}")
        print(f"    Mapper: {mapper_model}/{mapper_prompt}")
        
        # Load the norma results
        df_norma = pd.read_csv(csv_file)
        
        # Check for DC column (might be 'dc' or 'rule' or similar)
        dc_column = None
        for col in ['dc', 'rule', 'DC', 'Rule']:
            if col in df_norma.columns:
                dc_column = col
                break
        
        if dc_column is None:
            print(f"    Warning: No DC column found in {csv_file.name}")
            print(f"    Available columns: {df_norma.columns.tolist()}")
            continue
        
        dc_list = df_norma[dc_column].dropna().tolist()
        
        if not dc_list:
            print(f"    No DCs found in {csv_file.name}")
            continue
        
        # Parse all DCs from this file
        discovered_predicates = []
        for dc in dc_list:
            try:
                # Try LLM format parser first (uppercase AND)
                predicates = parse_dc_to_predicates_for_llm_baseline(str(dc))
                discovered_predicates.append(predicates)
            except Exception as e:
                # Try standard format parser (lowercase and)
                try:
                    predicates = parse_dc_to_predicates(str(dc))
                    discovered_predicates.append(predicates)
                except Exception as e2:
                    print(f"    Error parsing DC: {dc[:50]}... - {e}, {e2}")
                    continue

        for dc in discovered_predicates:
            print(f"      Parsed DC: {dc}")
        
        if not discovered_predicates:
            print(f"    No valid DCs parsed from {csv_file.name}")
            continue
        
        # Compute metrics for all discovered DCs
        metrics = compute_dc_metrics(all_dcs, discovered_predicates)
        
        norma_results.append({
            'dataset': dataset,
            'classifier_model': classifier_model,
            'classifier_prompt': classifier_prompt,
            'mapper_model': mapper_model,
            'mapper_prompt': mapper_prompt,
            'method': f"{classifier_model}_{classifier_prompt}_{mapper_model}_{mapper_prompt}",
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'tp': metrics['tp'],
            'fp': metrics['fp'],
            'fn': metrics['fn'],
            'gt_count': metrics['gt_count'],
            'discovered_count': len(discovered_predicates),
            'best_f1': False
        })

# Convert to DataFrame
norma_results_df = pd.DataFrame(norma_results)

if not norma_results_df.empty:
    # Mark best F1 for each dataset
    for dataset in norma_results_df['dataset'].unique():
        mask = norma_results_df['dataset'] == dataset
        if mask.sum() > 0:
            best_idx = norma_results_df[mask]['f1'].idxmax()
            norma_results_df.at[best_idx, 'best_f1'] = True
    
    # # Save to CSV
    # norma_results_df.to_csv(output_file, index=False)
    # print(f"\n{'='*60}")
    # print(f"Norma Results saved to: {output_file}")
    # print(f"Total rows: {len(norma_results_df)}")
    
    # Display summary
    print(f"\n{'='*60}")
    print("Summary by dataset:")
    for dataset in norma_results_df['dataset'].unique():
        dataset_df = norma_results_df[norma_results_df['dataset'] == dataset]
        print(f"\n{dataset}:")
        print(f"  Total configurations: {len(dataset_df)}")
        print(f"  Best F1: {dataset_df['f1'].max():.4f}")
        
        best_row = dataset_df[dataset_df['best_f1'] == True]
        if not best_row.empty:
            row = best_row.iloc[0]
            print(f"  Best: {row['method']}")
            print(f"    Classifier: {row['classifier_model']}/{row['classifier_prompt']}")
            print(f"    Mapper: {row['mapper_model']}/{row['mapper_prompt']}")
            print(f"    Precision: {row['precision']:.4f}, Recall: {row['recall']:.4f}, F1: {row['f1']:.4f}")
            print(f"    TP: {row['tp']}, FP: {row['fp']}, FN: {row['fn']}")
            print(f"    Discovered: {row['discovered_count']}, Ground Truth: {row['gt_count']}")
    print(f"{'='*60}")
    
    # Add Norma results to main_results
    for dataset in norma_results_df['dataset'].unique():
        dataset_norma = norma_results_df[(norma_results_df['dataset'] == dataset) & 
                                         (norma_results_df['best_f1'] == True)]
        if not dataset_norma.empty:
            row = dataset_norma.iloc[0]
            main_results.append({
                'dataset': dataset,
                'method': 'Norma',
                'precision': row['precision'],
                'recall': row['recall'],
                'f1': row['f1']
            })
else:
    print("No Norma results found")


Processing Norma results for dataset: restaurant-inspections
  Processing: restaurant-inspections_20260204_072448.csv
    Classifier: mistral-large-2512/few_shot
    Mapper: gpt-5/enhanced_with_domain_v2
      Parsed DC: {'t1.score<=13', 't1.score>=0', "t1.grade!='a'"}
      Parsed DC: {'t1.score<=27', 't1.score>=14', "t1.grade!='b'"}
      Parsed DC: {"t1.grade!='c'", 't1.score>=28'}
      Parsed DC: {'t1.days_to_next_inspection>365'}
      Parsed DC: {"t1.grade='a'", 't1.inspection_order=1', 't1.score>=0', 't1.grade_date!=t1.inspection_date', 't1.score<=13'}
      Parsed DC: {'t1.grade_date=t1.inspection_date', "t1.grade!='a'", 't1.inspection_order=1'}
      Parsed DC: {"t1.grade='a'", 't1.grade_date!=t1.inspection_date', "t1.inspection_typelike'%re-inspection%'"}
      Parsed DC: {'t1.inspection_date!=t1.grade_date', "t1.gradein('b','c')", "t1.inspection_typelike'%re-inspection%'"}
      Parsed DC: {'t1.camis=t2.camis', "t1.grade='a'", 't1.grade!=t2.grade', 't1.days_to_next_inspect

In [9]:
## Generate main results table and LaTeX output
# Create main results dataframe
main_results_df = pd.DataFrame(main_results)

# Save to CSV
if not main_results_df.empty:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_file = output_dir / f'main_results_{timestamp}.csv'
    main_results_df.to_csv(output_file, index=False)
    print(f"\n{'='*60}")
    print(f"Main results saved to: {output_file}")
    print(f"\nMain Results Table:")
    print(main_results_df.to_string(index=False))
    
    # Generate LaTeX table
    latex_df = main_results_df.copy()
    
    # Format numbers to 4 decimal places
    latex_df['precision'] = latex_df['precision'].apply(lambda x: f"{x:.4f}")
    latex_df['recall'] = latex_df['recall'].apply(lambda x: f"{x:.4f}")
    latex_df['f1'] = latex_df['f1'].apply(lambda x: f"{x:.4f}")
    
    # Generate LaTeX table
    latex_table = latex_df.to_latex(
        index=False,
        column_format='l|l|r|r|r',
        caption='Best DC Discovery Results by Method Category',
        label='tab:dc_discovery_results',
        escape=False
    )
    
    print(f"\n{'='*60}")
    print("LaTeX Table:")
    print("="*80)
    print(latex_table)
    print("="*80)
    
    # Save to file
    latex_file = output_dir / f'main_results_latex_{timestamp}.tex'
    with open(latex_file, 'w') as f:
        f.write(latex_table)
    print(f"\nLaTeX table saved to: {latex_file}")
else:
    print("No results to save")



Main results saved to: dc_discovery_baseline_results/main_results_20260206_072056.csv

Main Results Table:
               dataset                  method  precision   recall       f1
restaurant-inspections       DCDiscovery-Exact   0.000000 0.000000 0.000000
restaurant-inspections DCDiscovery-Approximate   0.000000 0.000000 0.000000
restaurant-inspections           LLM-AllAtOnce   0.750000 0.333333 0.461538
restaurant-inspections  LLM-SentenceBySentence   0.428571 0.333333 0.375000
restaurant-inspections                   Norma   0.555556 0.555556 0.555556

LaTeX Table:
\begin{table}
\centering
\caption{Best DC Discovery Results by Method Category}
\label{tab:dc_discovery_results}
\begin{tabular}{l|l|r|r|r}
\toprule
               dataset &                  method & precision & recall &     f1 \\
\midrule
restaurant-inspections &       DCDiscovery-Exact &    0.0000 & 0.0000 & 0.0000 \\
restaurant-inspections & DCDiscovery-Approximate &    0.0000 & 0.0000 & 0.0000 \\
restaurant-inspect

/var/folders/kt/v8ddhvyx5kv0zx8hg7_zkb4m0000gn/T/ipykernel_4975/3192306457.py:24: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex_table = latex_df.to_latex(
